# exp_e2_mpnet_multi — Semantic Graph Builder v2

**Phase 1a, embedder = `paraphrase-multilingual-mpnet-base-v2`, LLM = `deepseek-v32/latest` (API).**

Запускается из `exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/` — все пути разрешаются автоматически от `clustering_1/`.

Перед запуском: `export YANDEX_CLOUD_API_KEY=...`.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

In [2]:
import os, sys, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# layout: clustering_1/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/
EXP_DIR   = Path().resolve()
REPO_ROOT = EXP_DIR.parents[3]                 # clustering_1/
LLM_V2    = REPO_ROOT / 'llm_v2'

# put repo root on sys.path so `import llm_v2` works
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# guard: never expose llm_v2/ as flat path
if str(LLM_V2) in sys.path:
    sys.path.remove(str(LLM_V2))

from llm_v2.config_schema import load_config
config = load_config(EXP_DIR / 'config.yaml')

# expand ${YANDEX_CLOUD_API_KEY} etc. (no-op for local LLMs)
config.llm.api_key = os.path.expandvars(config.llm.api_key)
config.llm.base_url = os.path.expandvars(config.llm.base_url)
config.llm.folder = os.path.expandvars(config.llm.folder)

print('EXP_DIR  :', EXP_DIR)
print('REPO_ROOT:', REPO_ROOT)
print('LLM_V2   :', LLM_V2)
print()
print(config.model_dump_json(indent=2))

EXP_DIR  : /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi
REPO_ROOT: /home/platoon/graph/semantic-graph
LLM_V2   : /home/platoon/graph/semantic-graph/llm_v2

{
  "llm": {
    "provider": "api",
    "model_name": "deepseek-v32/latest",
    "max_new_tokens": 500,
    "temperature": 0.3,
    "device": "cpu",
    "load_in_8bit": false,
    "api_key": "${YANDEX_CLOUD_API_KEY}",
    "base_url": "https://ai.api.cloud.yandex.net/v1",
    "folder": "b1gpiug3vgbpe1cb4e5c",
    "instructions": ""
  },
  "embedding": {
    "model_name": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "device": "cuda"
  },
  "coreference": {
    "enabled": false,
    "prompt_file": "prompts/coreference_ru.txt",
    "context_sentences": 3,
    "window_sentences": 5
  },
  "extraction": {
    "prompt_file": "prompts/extraction_ru.txt",
    "chunk_size": 3,
    "overlap_size": 1
  },
  "normalization": {
    "enabled": true,
    "language": "ru"
  },
  "dedup

In [ ]:
config.llm.api_key = ""

In [4]:
from llm_v2.models.llm_client import LLMClient
from llm_v2.models.embedder import Embedder

llm = LLMClient(config.llm)
embedder = Embedder(config.embedding)
print(f'LLM loaded: {config.llm.model_name}')
print(f'Embedder loaded: {config.embedding.model_name} (dim={embedder.dim})')

/home/platoon/graph/graph_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-07 15:14:28,253 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
2026-05-07 15:14:28,666 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 15:14:28,711 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json "HTTP/1.1 200 OK"
2026-05-07 15:14:28,862 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP

LLM loaded: deepseek-v32/latest
Embedder loaded: sentence-transformers/paraphrase-multilingual-mpnet-base-v2 (dim=768)


In [5]:
from llm_v2.utils.io import load_text

input_path = Path(config.paths.input_text)
if not input_path.is_absolute():
    input_path = (LLM_V2 / input_path).resolve()
text = load_text(input_path)
print(f'Input: {input_path}')
print(f'Length: {len(text)} chars')
print(text[:500])

Input: /home/platoon/graph/semantic-graph/benchmark/final_bench/formated_fragment2.md
Length: 15466 chars
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.

Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.

В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам 


## [0] Preprocessing

In [6]:
from llm_v2.stages.preprocessing import preprocess

sentences = preprocess(text, language=config.normalization.language)
for s in sentences:
    print(f'  [{s.id}] {s.text}')

  [0] # Линейная классификация

Теперь давайте поговорим про задачу классификации.
  [1] Для начала будем говорить про бинарную классификацию на два класса.
  [2] Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.
  [3] Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.
  [4] В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$.
  [5] Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого.
  [6] **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой.
  [7] Выборка, для которой это возможно, называется линейно разделимой.
  [8] Увы, в реальной жизни такое встречается кр

## [1] Coreference Resolution

In [7]:
from llm_v2.stages.coreference import resolve_coreferences

resolved_text, sentences = resolve_coreferences(
    sentences, llm, config.coreference, base_dir=LLM_V2
)
print('Resolved text:')
print(resolved_text)
print(f'\nSentences after coref: {len(sentences)}')

Resolved text:
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда. Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$. В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$. Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого. **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой. Выборка, для которой это возможно, называется линейно разделимой. Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель

## [1.5] Chunking

In [8]:
from llm_v2.stages.chunking import build_chunks

chunks = build_chunks(sentences, config.extraction)
for c in chunks:
    print(f'  {c.id} (sents {c.sentence_ids}): {c.text[:80]}...')

  chunk_0 (sents [0, 1, 2]): # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для...
  chunk_1 (sents [2, 3, 4]): Обобщить эту задачу до задачи классификации на $K$ классов не составит большого ...
  chunk_2 (sents [4, 5, 6]): В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам буду...
  chunk_3 (sents [6, 7, 8]): **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: пол...
  chunk_4 (sents [8, 9, 10]): Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модел...
  chunk_5 (sents [10, 11, 12]): $$

<details>
<summary>Почему бы не решать задачу классификации как задачу регре...
  chunk_6 (sents [12, 13, 14]): Во вторых, ошибкой будет считаться предсказание, например, $5$ вместо $1$, хотя ...
  chunk_7 (sents [14, 15, 16]): </details>

Сконструируем теперь функционал ошибки так, чтобы он вышеперечисленн...
  chunk_8 (sents [16, 17, 18]): $$

Домножим обе части на $y_i$ и немного упростим:

$

## [2] Triplet Extraction

In [9]:
from llm_v2.stages.extraction import extract_triplets

raw_triplets = extract_triplets(chunks, llm, config.extraction, base_dir=LLM_V2)
print(f'Extracted {len(raw_triplets)} raw triplets:')
for t in raw_triplets:
    print(f'  {t.subject} | {t.relation} | {t.object}  [{t.chunk_id}]')

Extracting triplets: 100%|██████████| 56/56 [10:31<00:00, 11.28s/it]

Extracted 336 raw triplets:
  линейная классификация | является | задача классификации  [chunk_0]
  задача классификации | включает | бинарная классификация  [chunk_0]
  бинарная классификация | использует | два класса  [chunk_0]
  задача классификации | может быть обобщена до | классификация на K классов  [chunk_0]
  задача | обобщается до | классификация на K классов  [chunk_1]
  обобщение задачи | не составляет | большого труда  [chunk_1]
  таргеты y | кодируют | принадлежность к классу  [chunk_1]
  таргеты y | кодируют принадлежность к | положительный класс  [chunk_1]
  таргеты y | кодируют принадлежность к | отрицательный класс  [chunk_1]
  принадлежность | является принадлежностью множеству | {-1,1}  [chunk_1]
  x | является | векторы  [chunk_1]
  векторы | принадлежат пространству | ℝ^D  [chunk_1]
  договорённость | устанавливает обозначение для | классы  [chunk_1]
  метки {0,1} | встречаются в жизни | нередко  [chunk_1]
  мы | обозначаем | классы  [chunk_2]
  метки {0,1} | встр

## [3] Normalization

In [10]:
from llm_v2.stages.normalization import normalize_triplets

norm_triplets = normalize_triplets(raw_triplets, config.normalization)
print(f'Normalized {len(norm_triplets)} triplets:')
for t in norm_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

2026-05-07 15:25:40,472 [INFO] Loading dictionaries from /home/platoon/graph/graph_env/lib/python3.12/site-packages/pymorphy3_dicts_ru/data
2026-05-07 15:25:40,506 [INFO] format: 2.4, revision: 417150, updated: 2022-01-08T22:09:24.565962


Normalized 336 triplets:
  линейный классификация | являться | задача классификация
  задача классификация | включать | бинарный классификация
  бинарный классификация | использовать | два класс
  задача классификация | мочь быть обобщить до | классификация на k класс
  задача | обобщаться до | классификация на k класс
  обобщение задача | не составлять | большой труд
  таргет y | кодировать | принадлежность к класс
  таргет y | кодировать принадлежность к | положительный класс
  таргет y | кодировать принадлежность к | отрицательный класс
  принадлежность | являться принадлежность множество | {-1,1}
  x | являться | вектор
  вектор | принадлежать пространство | ℝ^D
  договорённость | устанавливать обозначение для | класс
  метка {0,1} | встречаться в жизнь | нередко
  мы | обозначать | класс
  метка {0,1} | встречаться в жизнь | вы
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | отделять | объект один класс
  плоскость | отделять | объект

## [4] Deduplication

In [11]:
from llm_v2.stages.deduplication import deduplicate_triplets

dedup_triplets = deduplicate_triplets(norm_triplets, embedder, config.deduplication)
print(f'After dedup: {len(norm_triplets)} -> {len(dedup_triplets)} triplets')
for t in dedup_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

After dedup: 336 -> 286 triplets
  линейный классификация | являться | задача классификация
  задача классификация | включать | бинарный классификация
  бинарный классификация | использовать | два класс
  задача классификация | мочь быть обобщить до | классификация на k класс
  задача | обобщаться до | классификация на k класс
  обобщение задача | не составлять | большой труд
  таргет y | кодировать | принадлежность к класс
  таргет y | кодировать принадлежность к | положительный класс
  принадлежность | являться принадлежность множество | {-1,1}
  x | являться | вектор
  вектор | принадлежать пространство | ℝ^D
  договорённость | устанавливать обозначение для | класс
  метка {0,1} | встречаться в жизнь | нередко
  мы | обозначать | класс
  метка {0,1} | встречаться в жизнь | вы
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | отделять | объект один класс
  в идеальный ситуация | найтись | плоскость
  плоскость | разделить | класс
  положит

## [5] Graph Assembly (raw)

In [12]:
from llm_v2.stages.graph_assembly import assemble_graph

raw_graph = assemble_graph(dedup_triplets, chunks, text, config)
print(f'Raw graph: {len(raw_graph.nodes)} nodes, {len(raw_graph.edges)} edges')
print('\nNodes:')
for n in raw_graph.nodes:
    print(f'  {n.id}: {n.label} ({len(n.mentions)} mentions)')
print('\nEdges:')
for e in raw_graph.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (w={e.weight})')

Raw graph: 320 nodes, 286 edges

Nodes:
  n0: линейный классификация (1 mentions)
  n1: задача классификация (4 mentions)
  n2: бинарный классификация (2 mentions)
  n3: два класс (1 mentions)
  n4: классификация на k класс (2 mentions)
  n5: задача (3 mentions)
  n6: обобщение задача (1 mentions)
  n7: большой труд (1 mentions)
  n8: таргет y (2 mentions)
  n9: принадлежность к класс (1 mentions)
  n10: положительный класс (4 mentions)
  n11: принадлежность (1 mentions)
  n12: {-1,1} (1 mentions)
  n13: x (1 mentions)
  n14: вектор (2 mentions)
  n15: ℝ^D (1 mentions)
  n16: договорённость (1 mentions)
  n17: класс (8 mentions)
  n18: метка {0,1} (2 mentions)
  n19: нередко (1 mentions)
  n20: мы (25 mentions)
  n21: вы (1 mentions)
  n22: линейный модель (5 mentions)
  n23: плоскость (5 mentions)
  n24: объект один класс (1 mentions)
  n25: в идеальный ситуация (1 mentions)
  n26: от плоскость (2 mentions)
  n27: отрицательный класс (1 mentions)
  n28: выборка (1 mentions)
  n29: лин

## [6] Clustering

In [13]:
from llm_v2.stages.clustering import cluster_graph, cluster_graph_multi, cluster_graph_all_methods
from llm_v2.utils.io import load_prompt

naming_prompt_path = Path(config.clustering.cluster_naming_prompt)
if not naming_prompt_path.is_absolute():
    naming_prompt_path = (LLM_V2 / naming_prompt_path).resolve()
naming_prompt = load_prompt(naming_prompt_path) if naming_prompt_path.exists() else None

if config.clustering.multi_method:
    multi = cluster_graph_all_methods(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    print('Multi-method clustering:')
    for method_name, mr in multi.methods.items():
        print(f'  {method_name}: {len(mr.param_labels)} variants')
        for lbl in mr.param_labels:
            g = mr.graphs[lbl]
            print(f'    {lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    agg = multi.methods['agglomerative']
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
elif config.clustering.is_multi_threshold:
    multi = cluster_graph_multi(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    agg = multi.methods['agglomerative']
    print(f'Multi-threshold: {len(agg.param_labels)} levels')
    for lbl in agg.param_labels:
        g = agg.graphs[lbl]
        print(f'  t={lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
else:
    clustered = cluster_graph(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )

print(f'\nClustered graph: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print('\nClustered Nodes:')
for n in clustered.nodes:
    print(f'  {n.id}: {n.label} (members={n.members}, size={n.size})')
print('\nClustered Edges:')
for e in clustered.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (size={e.size})')

Multi-method clustering:
  agglomerative: 10 variants
    0.250: 191 nodes, 219 edges
    0.322: 132 nodes, 169 edges
    0.394: 94 nodes, 135 edges
    0.467: 71 nodes, 118 edges
    0.539: 52 nodes, 94 edges
    0.611: 43 nodes, 83 edges
    0.683: 35 nodes, 74 edges
    0.756: 27 nodes, 64 edges
    0.828: 16 nodes, 50 edges
    0.900: 7 nodes, 21 edges
  kmeans: 4 variants
    k=10: 10 nodes, 40 edges
    k=25: 25 nodes, 64 edges
    k=40: 40 nodes, 78 edges
    k=55: 55 nodes, 102 edges
  hdbscan: 9 variants
    mcs=3,ms=1: 56 nodes, 96 edges
    mcs=3,ms=3: 74 nodes, 111 edges
    mcs=3,ms=5: 115 nodes, 144 edges
    mcs=5,ms=1: 68 nodes, 99 edges
    mcs=5,ms=3: 84 nodes, 114 edges
    mcs=5,ms=5: 99 nodes, 125 edges
    mcs=10,ms=1: 93 nodes, 105 edges
    mcs=10,ms=3: 84 nodes, 92 edges
    mcs=10,ms=5: 102 nodes, 109 edges

Clustered graph: 43 nodes, 83 edges

Clustered Nodes:
  c0: распределение бернулли (members=['n209', 'n239', 'n241', 'n242', 'n243', 'n244', 'n245', 'n246

## Save outputs

In [14]:
from llm_v2.utils.io import save_json, save_text

out = EXP_DIR / config.paths.output_dir
out.mkdir(parents=True, exist_ok=True)

save_text(resolved_text, out / 'coreference_resolved.txt')
save_json(raw_graph.model_dump(), out / 'raw_graph.json')
save_json(clustered.model_dump(), out / 'clustered_graph.json')

if config.clustering.multi_method or config.clustering.is_multi_threshold:
    save_json(multi.model_dump(), out / 'multi_clustered_graph.json')
    method_counts = {m: len(r.param_labels) for m, r in multi.methods.items()}
    print(f'Saved multi_clustered_graph.json (methods: {method_counts})')

print(f'Saved to {out}/')

Saved multi_clustered_graph.json (methods: {'agglomerative': 10, 'kmeans': 4, 'hdbscan': 9})
Saved to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/output/


## Benchmark vs ground-truth graph

In [15]:
from llm_v2.benchmark import (
    evaluate_graph,
    evaluate_multi_graph,
    load_clustered_graph,
    print_metrics,
    print_multi_metrics,
    best_variant,
    show_node_alignments,
    show_edge_alignments,
    multi_metrics_to_dict,
)

# GT inputs (absolute, robust to CWD)
gt_graph_path = REPO_ROOT / 'benchmark' / 'final_bench' / 'graph_clustered.json'
gt_text_path  = REPO_ROOT / 'benchmark' / 'final_bench' / 'formated_fragment2.md'

gt_graph = load_clustered_graph(gt_graph_path)
gt_text  = gt_text_path.read_text(encoding='utf-8')

# embedding context: prefer the coreference-resolved text the pipeline saw
source_text = resolved_text if resolved_text else gt_text

print(f'GT  : {len(gt_graph.nodes)} nodes, {len(gt_graph.edges)} edges  ({gt_graph_path})')
print(f'Pred: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print(f'Context text: {len(source_text)} chars')

TAU_NODE = 0.6
TAU_EDGE = 0.6
BETA = 1.0
NODE_WEIGHT = 0.6
EDGE_WEIGHT = 0.4
NODE_WINDOW = 300
EDGE_WINDOW = 400
TOP_K = 10

GT  : 55 nodes, 51 edges  (/home/platoon/graph/semantic-graph/benchmark/final_bench/graph_clustered.json)
Pred: 43 nodes, 83 edges
Context text: 15418 chars


In [16]:
metrics = evaluate_graph(
    pred=clustered,
    gt=gt_graph,
    source_text=source_text,
    embedder=embedder,
    tau_node=TAU_NODE,
    tau_edge=TAU_EDGE,
    beta=BETA,
    node_weight=NODE_WEIGHT,
    edge_weight=EDGE_WEIGHT,
    node_window=NODE_WINDOW,
    edge_window=EDGE_WINDOW,
)

print_metrics(metrics)
metrics.summary()

Batches: 100%|██████████| 1/1 [00:00<00:00, 197.85it/s]

GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.2823

Nodes (pred=43, gt=55, matched=37, tau=0.6, beta=1):
  TP(soft)  = 15.1909
  precision = 0.3533
  recall    = 0.2762
  F1        = 0.3100
Edges (pred=83, gt=51, matched=47, tau=0.6, beta=1):
  TP(soft)  = 16.1334
  precision = 0.1944
  recall    = 0.3163
  F1        = 0.2408


{'graph_score': 0.2823296252071876,
 'node_weight': 0.6,
 'edge_weight': 0.4,
 'nodes': {'precision': 0.35327580293943717,
  'recall': 0.27619744593446904,
  'f_beta': 0.3100175413550163,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 15.190859526395798,
  'pred_count': 43,
  'gt_count': 55,
  'matched_count': 37},
 'edges': {'precision': 0.1943789074219853,
  'recall': 0.3163421434514663,
  'f_beta': 0.24079775098544448,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 16.13344931602478,
  'pred_count': 83,
  'gt_count': 51,
  'matched_count': 47},
 'pred_structure': {'n_nodes': 43,
  'n_edges': 83,
  'density': 0.04595791805094131,
  'n_components': 3,
  'n_isolated': 2,
  'component_sizes': [41, 1, 1],
  'component_size_min': 1,
  'component_size_max': 41,
  'component_size_mean': 14.333333333333334,
  'component_size_quantiles': {'q25': 1.0,
   'q50': 1.0,
   'q75': 21.0,
   'q90': 33.0}},
 'gt_structure': {'n_nodes': 55,
  'n_edges': 51,
  'density': 0.01717171717171717,
  'n_components': 6,
  'n_isola

In [17]:
show_node_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched node pairs: 37 / min(43, 55)=43

Top 10 matched (by quality q):
  [q=1.000]  'стохастический градиентный спуск'  ↔  'стохастический градиентный спуск'
  [q=0.989]  'число ошибка классификатор'  ↔  'число ошибок классификатора'
  [q=0.970]  'разделять поверхность'  ↔  'разделяющая поверхность'
  [q=0.949]  'выборка'  ↔  'выборка'
  [q=0.749]  'положение плоскость'  ↔  'вектор'
  [q=0.718]  'тут'  ↔  'логистическая регрессия'
  [q=0.653]  'знак'  ↔  'MSE'
  [q=0.641]  'в случай > 0'  ↔  '$\\nabla_w L(w,x,y)=2\\lambda w+\\sum_i\\begin{cases}0, & 1-y_i\\langle w,x_i\\rangle\\le 0, \\\\ -y_i x_i, & 1-y_i\\langle w,x_i\\rangle>0.\\end{cases}$'
  [q=0.633]  'вещественный число'  ↔  'регрессия'
  [q=0.566]  'градиентный спуск'  ↔  '$L(w,X,y)=-\\sum_i\\left(y_i\\log(\\sigma(\\langle w,x_i\\rangle))+(1-y_i)\\log(\\sigma(-\\langle w,x_i\\rangle))\\right)$'

Bottom 10 matched:
  [q=0.237]  'ответ'  ↔  'правдоподобие $p(y\\mid X,w)$'
  [q=0.221]  'класс модель'  ↔  'линейная модель'
  [q=0.

In [18]:
show_edge_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched edge pairs: 47 / min(83, 51)=51

Top 10 matched (by quality q):
  [q=0.872]  вы —[являться противоположный]→ класс модель
           ↔  выборка —[может обладать свойством]→ линейная разделимость
  [q=0.849]  распределение бернулли —[являться]→ в наш метод
           ↔  правдоподобие $p(y\mid X,w)$ —[основано на]→ распределение Бернулли
  [q=0.722]  мы —[перейти к]→ произведение
           ↔  правдоподобие $p(y\mid X,w)$ —[для распределения Бернулли равно]→ $p(y\mid X,w)=\prod_i p_i^{y_i}(1-p_i)^{1-y_i}$
  [q=0.707]  мы —[важный]→ знак
           ↔  регрессия —[минимизирует]→ MSE
  [q=0.687]  мы —[использовать]→ градиентный спуск
           ↔  функция потерь $L(w,X,y)$ —[равна]→ $L(w,X,y)=-\sum_i\left(y_i\log(\sigma(\langle w,x_i\rangle))+(1-y_i)\log(\sigma(-\langle w,x_i\rangle))\right)$
  [q=0.664]  вещественный число —[появляться из]→ пример
           ↔  регрессия —[может быть наивным подходом к]→ задача классификации
  [q=0.618]  мы —[хотеть минимизировать]→ число ошибка кл

In [19]:
save_json(metrics.summary(), out / 'benchmark_metrics.json')
print(f'Saved benchmark_metrics.json to {out}/')

Saved benchmark_metrics.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/output/


## Benchmark — multi-method / multi-threshold sweep

In [20]:
is_multi = config.clustering.multi_method or config.clustering.is_multi_threshold

if not is_multi:
    print('Skipped: multi-method / multi-threshold not enabled in config')
    multi_metrics = None
else:
    total = sum(len(mr.graphs) for mr in multi.methods.values())
    print(f'Evaluating {total} configurations...')
    multi_metrics = evaluate_multi_graph(
        multi=multi,
        gt=gt_graph,
        source_text=source_text,
        embedder=embedder,
        tau_node=TAU_NODE,
        tau_edge=TAU_EDGE,
        beta=BETA,
        node_weight=NODE_WEIGHT,
        edge_weight=EDGE_WEIGHT,
        node_window=NODE_WINDOW,
        edge_window=EDGE_WINDOW,
    )
    print(f'Done: {sum(len(v) for v in multi_metrics.values())} variants evaluated')

Evaluating 23 configurations...


Batches: 100%|██████████| 1/1 [00:00<00:00, 185.20it/s]


Done: 23 variants evaluated


In [21]:
if multi_metrics:
    print_multi_metrics(multi_metrics, sort_by='graph_score')

method        param                  pred_n pred_e  matched_n  matched_e    P_n    R_n    F_n    P_e    R_e    F_e   graph
--------------------------------------------------------------------------------------------------------------------------
agglomerative 0.467                      71    118         48         51  0.319  0.411  0.359  0.196  0.453  0.273  0.3247
agglomerative 0.394                      94    135         50         51  0.277  0.473  0.350  0.176  0.465  0.255  0.3117
agglomerative 0.683                      35     74         31         48  0.406  0.258  0.315  0.249  0.361  0.295  0.3073
agglomerative 0.539                      52     94         42         51  0.334  0.316  0.325  0.213  0.393  0.276  0.3054
agglomerative 0.322                     132    169         52         51  0.232  0.558  0.328  0.150  0.496  0.230  0.2888
agglomerative 0.611                      43     83         37         47  0.353  0.276  0.310  0.194  0.316  0.241  0.2823
agglomerative 0.

In [22]:
if multi_metrics:
    method, param, best_m = best_variant(multi_metrics, by='graph_score')
    best_graph = multi.methods[method].graphs[param]
    print(f'Best variant: method={method}, param={param}')
    print(f'  graph: {len(best_graph.nodes)} nodes, {len(best_graph.edges)} edges')
    print()
    print_metrics(best_m)

Best variant: method=hdbscan, param=mcs=5,ms=1
  graph: 68 nodes, 99 edges

GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.3414

Nodes (pred=68, gt=55, matched=50, tau=0.6, beta=1):
  TP(soft)  = 22.8781
  precision = 0.3364
  recall    = 0.4160
  F1        = 0.3720
Edges (pred=99, gt=51, matched=51, tau=0.6, beta=1):
  TP(soft)  = 22.1554
  precision = 0.2238
  recall    = 0.4344
  F1        = 0.2954


In [23]:
if multi_metrics:
    save_json(multi_metrics_to_dict(multi_metrics), out / 'benchmark_metrics_multi.json')
    print(f'Saved benchmark_metrics_multi.json to {out}/')

Saved benchmark_metrics_multi.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/output/
